[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/11_LangGraph/notebooks/03_agentic_rag_capstone.ipynb)

# Capstone — Self-Correcting Agentic RAG

This notebook builds **nothing new at the retrieval layer.** `HybridIndex`, `Reranker`, and
the exact prompts (`ANSWER_PROMPT`, `CONDENSE_PROMPT`) are **imported directly** from
`10_RAG/notebooks/production_rag_chatbot/rag_pipeline.py` — the class Module 10 built and
shipped. Nothing about chunking, dense+BM25 fusion, RRF, or cross-encoder reranking gets
retaught here.

What LangGraph adds is everything a straight-line chain structurally can't do:

| `ProductionRAGChatbot.chat()` (Module 10) | This capstone (Module 11) |
|---|---|
| retrieve → rerank → guardrail → generate, always in that order | retrieve → **grade** the results, and only proceed if they're actually sufficient |
| a weak retrieval just gets refused | a weak retrieval triggers a **query rewrite** and a **second attempt** |
| generation is trusted at face value | generation is **self-checked for groundedness** before it's shown to anyone |
| "I don't know" is the only fallback | after retries are exhausted, the graph **pauses and asks a human** (`interrupt()`) instead of guessing or flatly refusing |
| memory = a Python list on the instance | memory = a **checkpointer**, durable and resumable, keyed by `thread_id` |

## The graph

```
condense -> retrieve -> grade_documents --sufficient--> generate -> check_groundedness --grounded--> finalize -> END
               ^              |                             ^                |
               |         insufficient                       |          not grounded
               |         (retries left)                 regenerate      (retries left)
               |              v                             |                v
               +------- rewrite_query                       +---------- regenerate
                              |
                     (retries exhausted, either check)
                              v
                       human_escalation --interrupt()--> finalize -> END
```

## 0. Install dependencies

In [ ]:
%pip install -q langgraph>=0.6 langchain>=1.0 langchain-openai langchain-community \
    langchain-text-splitters langchain-pymupdf4llm pypdf docx2txt \
    sentence-transformers bm25s PyStemmer chromadb python-dotenv "numpy<2"

## 1. Setup — import the Module 10 pipeline, don't reimplement it

`rag_pipeline.py` lives at `10_RAG/notebooks/production_rag_chatbot/`. We add that directory
to `sys.path` and import straight from it, so this notebook and the Module 10 capstone share
one implementation instead of drifting apart.

In [ ]:
import warnings, os, sys
warnings.filterwarnings("ignore")
import logging
for _n in ("httpx", "openai", "httpcore", "sentence_transformers", "transformers", "chromadb"):
    logging.getLogger(_n).setLevel(logging.ERROR)

from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent.parent / ".env")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

RAG_PIPELINE_DIR = Path.cwd().parent.parent / "10_RAG" / "notebooks" / "production_rag_chatbot"
sys.path.insert(0, str(RAG_PIPELINE_DIR))

from rag_pipeline import (
    HybridIndex, Reranker, load_document, chunk_documents,
    format_sources, ANSWER_PROMPT, CONDENSE_PROMPT,
)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Imported HybridIndex, Reranker, and prompts from Module 10's rag_pipeline.py -- unmodified.")

## 2. Build the index — same sample report as Module 10's capstone

Reusing `sample_report.pdf` from `10_RAG/notebooks/data/` so the retrieval behavior here is
directly comparable to what you already saw in Notebook 13. Swap in your own document the
same way that notebook did.

In [ ]:
DATA_DIR = Path.cwd().parent.parent / "10_RAG" / "notebooks" / "data"
report_path = DATA_DIR / "sample_report.pdf"

pages = load_document(str(report_path))
chunks = chunk_documents(pages, chunk_size=500, chunk_overlap=80)

index = HybridIndex()
index.build(chunks)
reranker = Reranker()

print(f"Indexed {len(chunks)} chunks from {report_path.name}")

## 3. State — the shared scratchpad every node reads and writes

`messages` uses the `add_messages` reducer so multi-turn chat history accumulates instead of
being overwritten every turn — this is what makes the graph's memory conversational, not just
single-shot.

In [ ]:
from typing import Annotated, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class AgentState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    standalone_question: str
    reranked: list
    grade: str                 # "sufficient" | "insufficient"
    rewrite_count: int
    answer: str
    groundedness: str          # "grounded" | "not_grounded" | "human_provided" | "refused"
    regenerate_count: int


MAX_REWRITES = 2
MAX_REGENERATE = 1

## 4. Nodes

Each node is a small, testable function. Notice how much of this is Module 10's own logic —
`condense` is the condense-question pattern, `retrieve` is `HybridIndex.search` +
`Reranker.rerank` verbatim, `generate` reuses `ANSWER_PROMPT` and `format_sources` unchanged.
The *new* nodes are `grade_documents`, `rewrite_query`, `check_groundedness`, and
`human_escalation` — the self-correction loop a chain couldn't have.

In [ ]:
def condense(state: AgentState) -> dict:
    """Module 10's condense-question pattern, now reading history from graph state
    instead of a hand-maintained self.history list."""
    question = state["messages"][-1].content
    history = state["messages"][:-1]
    if not history:
        standalone = question
    else:
        history_text = "\n".join(f"{m.type}: {m.content}" for m in history[-6:])
        standalone = llm.invoke(
            CONDENSE_PROMPT.format(history=history_text, question=question)
        ).content.strip()
    return {"standalone_question": standalone, "rewrite_count": 0, "regenerate_count": 0}


def retrieve(state: AgentState) -> dict:
    """Module 10's HybridIndex + Reranker, called exactly as ProductionRAGChatbot.chat() does."""
    candidates = index.search(state["standalone_question"], k=8)
    reranked = reranker.rerank(state["standalone_question"], candidates, top_n=4)
    return {"reranked": reranked}


GRADE_PROMPT = """Are the SOURCES below sufficient to answer the QUESTION well? Answer with
exactly one word: SUFFICIENT or INSUFFICIENT.

QUESTION: {question}

SOURCES:
{sources}"""


def grade_documents(state: AgentState) -> dict:
    """The self-correction step Module 10 never had: judge the retrieval before trusting it."""
    sources = format_sources(state["reranked"]) if state["reranked"] else "(nothing retrieved)"
    verdict = llm.invoke(
        GRADE_PROMPT.format(question=state["standalone_question"], sources=sources)
    ).content.strip().upper()
    grade = "insufficient" if "INSUFFICIENT" in verdict else "sufficient"
    return {"grade": grade}


REWRITE_PROMPT = """The QUESTION below did not retrieve sufficient sources from the knowledge
base. Rewrite it to be more specific and retrieval-friendly. Return ONLY the rewritten
question, nothing else.

QUESTION: {question}"""


def rewrite_query(state: AgentState) -> dict:
    rewritten = llm.invoke(REWRITE_PROMPT.format(question=state["standalone_question"])).content.strip()
    return {"standalone_question": rewritten, "rewrite_count": state.get("rewrite_count", 0) + 1}


def generate(state: AgentState) -> dict:
    """Module 10's ANSWER_PROMPT and format_sources, unmodified."""
    history_text = "\n".join(f"{m.type}: {m.content}" for m in state["messages"][:-1][-6:])
    prompt = ANSWER_PROMPT.format(
        sources=format_sources(state["reranked"]), history=history_text,
        question=state["standalone_question"],
    )
    answer = llm.invoke(prompt).content.strip()
    return {"answer": answer}


GROUNDEDNESS_PROMPT = """Does the ANSWER rely ONLY on facts present in the SOURCES, with no
invented details? Answer with exactly one word: GROUNDED or NOT_GROUNDED.

SOURCES:
{sources}

ANSWER:
{answer}"""


def check_groundedness(state: AgentState) -> dict:
    """A formal, resumable version of Module 10's rerank-score guardrail --
    this time checking the GENERATED TEXT itself, the same idea RAGAS faithfulness
    (Notebook 09) scores, but as a live gate instead of an offline metric."""
    verdict = llm.invoke(
        GROUNDEDNESS_PROMPT.format(sources=format_sources(state["reranked"]), answer=state["answer"])
    ).content.strip().upper()
    groundedness = "not_grounded" if "NOT_GROUNDED" in verdict else "grounded"
    return {"groundedness": groundedness}


def regenerate_bump(state: AgentState) -> dict:
    return {"regenerate_count": state.get("regenerate_count", 0) + 1}


def human_escalation(state: AgentState) -> dict:
    """The formal version of the human-in-the-loop pause Module 10 (S10f) only gestured at.
    Retries are exhausted -- pause the graph and hand the decision to a person instead of
    guessing."""
    guidance = interrupt({
        "reason": "Could not retrieve/generate a grounded answer after retrying.",
        "question": state["standalone_question"],
        "best_effort_answer": state.get("answer"),
        "rewrite_count": state.get("rewrite_count", 0),
        "regenerate_count": state.get("regenerate_count", 0),
    })
    if guidance and guidance.get("human_answer"):
        return {"answer": guidance["human_answer"], "groundedness": "human_provided"}
    return {
        "answer": "I don't have enough grounded information in this document to answer "
                   "confidently, and no human guidance was provided.",
        "groundedness": "refused",
    }


def finalize(state: AgentState) -> dict:
    from langchain_core.messages import AIMessage
    return {"messages": [AIMessage(content=state["answer"])]}

## 5. Routers — the conditional edges that make this a *graph*, not a chain

In [ ]:
def route_after_grade(state: AgentState) -> str:
    if state["grade"] == "sufficient":
        return "generate"
    if state.get("rewrite_count", 0) < MAX_REWRITES:
        return "rewrite_query"
    return "human_escalation"


def route_after_groundedness(state: AgentState) -> str:
    if state["groundedness"] == "grounded":
        return "finalize"
    if state.get("regenerate_count", 0) < MAX_REGENERATE:
        return "regenerate_bump"
    return "human_escalation"

## 6. Wire the graph and compile with a checkpointer

In [ ]:
builder = StateGraph(AgentState)
for name, fn in [
    ("condense", condense), ("retrieve", retrieve), ("grade_documents", grade_documents),
    ("rewrite_query", rewrite_query), ("generate", generate),
    ("check_groundedness", check_groundedness), ("regenerate_bump", regenerate_bump),
    ("human_escalation", human_escalation), ("finalize", finalize),
]:
    builder.add_node(name, fn)

builder.add_edge(START, "condense")
builder.add_edge("condense", "retrieve")
builder.add_edge("retrieve", "grade_documents")
builder.add_conditional_edges("grade_documents", route_after_grade, {
    "generate": "generate", "rewrite_query": "rewrite_query", "human_escalation": "human_escalation",
})
builder.add_edge("rewrite_query", "retrieve")
builder.add_edge("generate", "check_groundedness")
builder.add_conditional_edges("check_groundedness", route_after_groundedness, {
    "finalize": "finalize", "regenerate_bump": "regenerate_bump", "human_escalation": "human_escalation",
})
builder.add_edge("regenerate_bump", "generate")
builder.add_edge("human_escalation", "finalize")
builder.add_edge("finalize", END)

checkpointer = InMemorySaver()
agentic_rag = builder.compile(checkpointer=checkpointer)

print(agentic_rag.get_graph().draw_mermaid())

## 7. Run it — four turns, one thread, watching the trace each time

A small helper prints which nodes fired (`stream_mode="updates"`) so the self-correction
path is visible, not just the final answer — exactly the "assess the trace, not just the
answer" principle from the module README.

> Grading and rewriting are done by a live LLM call, so the exact retry path can vary
> run to run — that's expected, and worth calling out to students as a real property of
> agentic systems, not a bug in this notebook.

In [ ]:
from langchain_core.messages import HumanMessage

def ask(question, thread_id="capstone-demo"):
    config = {"configurable": {"thread_id": thread_id}}
    print(f"\n=== Q: {question}  (thread={thread_id}) ===")
    result = None
    for update in agentic_rag.stream({"messages": [HumanMessage(content=question)]}, config, stream_mode="updates"):
        for node_name, node_output in update.items():
            if node_name == "__interrupt__":
                print(f"  [PAUSED] human_escalation ->", node_output[0].value)
                return config, None
            shown = {k: v for k, v in node_output.items() if k != "messages"}
            print(f"  [{node_name}] {shown}")
        result = update
    final_state = agentic_rag.get_state(config).values
    print("ANSWER:", final_state.get("answer"))
    return config, final_state


# Turn 1 -- a normal, well-scoped question. Expect: retrieve -> grade(sufficient) -> generate
#          -> check(grounded) -> finalize, no retries.
ask("What were the total shipments mentioned in the report?")

In [ ]:
# Turn 2 -- deliberately vague, likely to fail the first grading pass and trigger a rewrite.
ask("Tell me about the numbers.", thread_id="capstone-demo")

In [ ]:
# Turn 3 -- genuinely out of scope. Expect retries to exhaust and human_escalation to PAUSE
# the graph with interrupt().
config3, state3 = ask("What is the capital of France?", thread_id="capstone-demo")

In [ ]:
# Resume Turn 3 as the human: either supply the real answer or approve the refusal.
resumed = agentic_rag.invoke(
    Command(resume={"human_answer": None}),   # None -> agent falls back to its honest refusal
    config3,
)
print("Resumed answer:", resumed["messages"][-1].content)

In [ ]:
# Turn 4 -- a pronoun follow-up. Only resolvable correctly if `condense` can see the SAME
# thread's history -- proving the checkpointer memory works across turns, not just within one.
ask("What about its risks or limitations?", thread_id="capstone-demo")

## 8. Ship it

The exact graph above is exported as a standalone module at
[`../capstone_agentic_rag/graph.py`](../capstone_agentic_rag/graph.py), imported by a
Streamlit chat app at [`../capstone_agentic_rag/app.py`](../capstone_agentic_rag/app.py) that
adds a live trace panel (every node that fired, retry counts, groundedness verdict) and a
human-approval box that fires whenever `human_escalation` pauses the graph. Run it with:

```bash
cd ../capstone_agentic_rag
streamlit run app.py
```

## Summary

- **Nothing about retrieval was rebuilt.** `HybridIndex`, `Reranker`, `ANSWER_PROMPT`,
  `CONDENSE_PROMPT` came straight from Module 10's `rag_pipeline.py`.
- **LangGraph's entire contribution is control flow the chain couldn't express**: a retry
  loop on weak retrieval (`grade_documents` → `rewrite_query`), a retry loop on ungrounded
  generation (`check_groundedness` → `regenerate_bump`), and a real pause-and-resume escape
  hatch to a human (`human_escalation` via `interrupt()`) instead of a hardcoded refusal
  string.
- **Memory is now durable and inspectable**, not a list on an instance —
  `agentic_rag.get_state(config)` shows you the exact state at any point, and the same
  `thread_id` resumes a conversation across any gap in time.

This is the pattern to reuse for almost any "agent" a student builds next: take a chain that
already works, identify the one or two places it needs to retry, branch, or wait for a human,
and wrap *just those transitions* in a graph — not the whole pipeline from scratch.